In [ ]:
# ── Install everything ────────────────────────────────────────────────────────
import subprocess
subprocess.run(["apt-get", "install", "-y", "openjdk-11-jdk"], capture_output=True)
subprocess.run(["pip", "install", "pyspark==3.5.3", "boto3", "-q"], check=True)

# ── Set up Java and Spark ─────────────────────────────────────────────────────
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Ecommerce").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready ✅")

# ── Download from S3 ──────────────────────────────────────────────────────────
import boto3

AWS_ACCESS_KEY = "YOUR_ACCESS_KEY_HERE"
AWS_SECRET_KEY = "YOUR_SECRET_KEY_HERE"
S3_BUCKET = "YOUR-BUCKET-NAME"

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="us-east-1"
)

os.makedirs("/content/bronze/orders", exist_ok=True)

response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="bronze/orders/")

downloaded = 0
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith("/"):
        continue
    local_path = f"/content/{key}"
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(S3_BUCKET, key, local_path)
    downloaded += 1
    print(f"✅ {key}")

print(f"\n{downloaded} files downloaded")

# ── Read with PySpark ─────────────────────────────────────────────────────────
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/bronze/orders/")

df_orders.show(5)
print(f"Total rows: {df_orders.count()}")

Spark 3.5.3 ready ✅
✅ bronze/orders/date=2026-05-01/orders_2026-05-01.csv
✅ bronze/orders/date=2026-05-02/orders_2026-05-02.csv
✅ bronze/orders/date=2026-05-03/orders_2026-05-03.csv
✅ bronze/orders/date=2026-05-04/orders_2026-05-04.csv
✅ bronze/orders/date=2026-05-05/orders_2026-05-05.csv
✅ bronze/orders/date=2026-05-06/orders_2026-05-06.csv
✅ bronze/orders/date=2026-05-07/orders_2026-05-07.csv
✅ bronze/orders/date=2026-05-08/orders_2026-05-08.csv

8 files downloaded
+----------------+---------+----------+--------------------+-------------+--------+----------+------------+-----------+--------+---------+--------------+----------------+-------------------+--------------------+----------+
|        order_id|  user_id|product_id|        product_name|     category|quantity|unit_price|discount_pct|total_price|currency|   status|payment_method|shipping_country|    order_timestamp| ingestion_timestamp|      date|
+----------------+---------+----------+--------------------+-------------+--------

In [ ]:
from pyspark.sql import functions as F

print("Starting Bronze → Silver transformation...")
print(f"Bronze row count: {df_orders.count()}")

# ── Step 1: Fix column types ───────────────────────────────────────────────
# order_timestamp is currently a string like "2026-05-06 15:40:42"
# We convert it to a real timestamp so we can do date calculations
df_silver = df_orders \
    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp", "yyyy-MM-dd HH:mm:ss")
    ) \
    .withColumn(
        "ingestion_timestamp",
        F.to_timestamp("ingestion_timestamp")
    )

# ── Step 2: Add useful derived columns ────────────────────────────────────
# Extract date and hour from timestamp — useful for analytics
df_silver = df_silver \
    .withColumn("order_date", F.to_date("order_timestamp")) \
    .withColumn("order_hour", F.hour("order_timestamp")) \
    .withColumn("order_month", F.month("order_timestamp"))

# ── Step 3: Data quality — remove bad rows ────────────────────────────────
# In production, bad rows go to a quarantine table
# For now we just remove them and count how many we dropped

total_before = df_silver.count()

df_silver = df_silver \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("user_id").isNotNull()) \
    .filter(F.col("total_price") > 0) \
    .filter(F.col("quantity") > 0)

total_after = df_silver.count()
dropped = total_before - total_after

print(f"Rows before quality check: {total_before}")
print(f"Rows after quality check:  {total_after}")
print(f"Rows dropped (bad data):   {dropped}")

# ── Step 4: Drop columns we don't need in Silver ──────────────────────────
# "date" was auto-added by Spark from folder name — redundant now
# we have order_date which is cleaner
df_silver = df_silver.drop("date")

# ── Step 5: Show the result ───────────────────────────────────────────────
print("\nSilver layer sample:")
df_silver.select(
    "order_id",
    "user_id",
    "total_price",
    "order_timestamp",
    "order_date",
    "order_hour",
    "status"
).show(5)

df_silver.printSchema()
print(f"\n✅ Bronze → Silver transformation complete")

Starting Bronze → Silver transformation...
Bronze row count: 1600
Rows before quality check: 1600
Rows after quality check:  1600
Rows dropped (bad data):   0

Silver layer sample:
+----------------+---------+-----------+-------------------+----------+----------+---------+
|        order_id|  user_id|total_price|    order_timestamp|order_date|order_hour|   status|
+----------------+---------+-----------+-------------------+----------+----------+---------+
|ORD-202605060000|USR-00250|      654.3|2026-05-06 15:40:42|2026-05-06|        15|completed|
|ORD-202605060001|USR-00459|      36.91|2026-05-06 12:56:02|2026-05-06|        12|completed|
|ORD-202605060002|USR-00401|    1374.66|2026-05-06 01:50:08|2026-05-06|         1|cancelled|
|ORD-202605060003|USR-00446|       91.4|2026-05-06 20:28:44|2026-05-06|        20|completed|
|ORD-202605060004|USR-00446|     175.82|2026-05-06 05:43:02|2026-05-06|         5|completed|
+----------------+---------+-----------+-------------------+----------+----

In [ ]:
# Write the cleaned Silver data
df_silver.write \
    .mode("overwrite") \
    .parquet("/content/silver/orders/")

print(f"✅ Silver layer written")
print(f"Total rows: {df_silver.count()}")

✅ Silver layer written
Total rows: 1600


In [ ]:
print("Building Gold aggregations...\n")

# ── Gold Table 1: Revenue by Category ────────────────────────────────────────
# Business question: which category drives the most revenue?
# SQL equivalent: SELECT category, SUM(total_price), COUNT(*) FROM silver GROUP BY category

gold_category = df_silver \
    .filter(F.col("status") == "completed") \
    .groupBy("category") \
    .agg(
        F.round(F.sum("total_price"), 2).alias("total_revenue"),
        F.count("order_id").alias("total_orders"),
        F.round(F.avg("total_price"), 2).alias("avg_order_value")
    ) \
    .orderBy(F.desc("total_revenue"))

print("💰 Revenue by Category:")
gold_category.show()

# ── Gold Table 2: Orders by Status ───────────────────────────────────────────
# Business question: what percentage of orders are completed vs cancelled?
# SQL equivalent: SELECT status, COUNT(*) FROM silver GROUP BY status

gold_status = df_silver \
    .groupBy("status") \
    .agg(
        F.count("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("total_orders"))

print("📦 Orders by Status:")
gold_status.show()

# ── Gold Table 3: Daily Revenue ───────────────────────────────────────────────
# Business question: how is revenue trending day by day?
# SQL equivalent: SELECT order_date, SUM(total_price), COUNT(*) FROM silver GROUP BY order_date

gold_daily = df_silver \
    .filter(F.col("status") == "completed") \
    .groupBy("order_date") \
    .agg(
        F.round(F.sum("total_price"), 2).alias("daily_revenue"),
        F.count("order_id").alias("total_orders")
    ) \
    .orderBy("order_date")

print("📈 Daily Revenue Trend:")
gold_daily.show()

print("✅ All Gold tables built")

Building Gold aggregations...

💰 Revenue by Category:
+-------------+-------------+------------+---------------+
|     category|total_revenue|total_orders|avg_order_value|
+-------------+-------------+------------+---------------+
|     Clothing|    121556.28|         141|          862.1|
|        Books|    113358.39|         136|         833.52|
|       Sports|    106103.93|         133|         797.77|
|Home & Garden|     93673.56|         144|         650.51|
|  Electronics|     92783.26|         123|         754.34|
|       Beauty|     71796.05|         131|         548.06|
+-------------+-------------+------------+---------------+

📦 Orders by Status:
+---------+------------+
|   status|total_orders|
+---------+------------+
|completed|         808|
|  pending|         270|
| refunded|         269|
|cancelled|         253|
+---------+------------+

📈 Daily Revenue Trend:
+----------+-------------+------------+
|order_date|daily_revenue|total_orders|
+----------+-------------+-----

In [ ]:
import boto3
import os

# We'll upload everything in /content/silver/ and /content/gold/
# back to your S3 bucket

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="us-east-1"
)

def upload_folder_to_s3(local_folder, s3_prefix):
    """
    Walks through every file in local_folder
    and uploads it to S3 under s3_prefix.
    Same structure, just different location.
    """
    uploaded = 0
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            # Skip Spark's internal metadata files
            if file.startswith(".") or file == "_SUCCESS":
                continue

            local_path = os.path.join(root, file)

            # Build the S3 key by replacing /content/ with nothing
            s3_key = s3_prefix + local_path.replace(local_folder, "")

            s3.upload_file(local_path, S3_BUCKET, s3_key)
            uploaded += 1
            print(f"✅ Uploaded: {s3_key}")

    print(f"\nTotal files uploaded: {uploaded}")

# Upload Silver orders
print("Uploading Silver orders to S3...")
upload_folder_to_s3("/content/silver/orders/", "silver/orders/")

# Write Gold tables first, then upload
print("\nWriting Gold tables locally...")

gold_category.write.mode("overwrite").parquet("/content/gold/revenue_by_category/")
gold_status.write.mode("overwrite").parquet("/content/gold/orders_by_status/")
gold_daily.write.mode("overwrite").parquet("/content/gold/daily_revenue/")

print("\nUploading Gold tables to S3...")
upload_folder_to_s3("/content/gold/revenue_by_category/", "silver/gold/revenue_by_category/")
upload_folder_to_s3("/content/gold/orders_by_status/", "gold/orders_by_status/")
upload_folder_to_s3("/content/gold/daily_revenue/", "gold/daily_revenue/")

print("\n🎉 Silver and Gold layers written to S3")

Uploading Silver orders to S3...
✅ Uploaded: silver/orders/part-00000-4b851e0b-6082-4265-ac6a-0ed4972b8351-c000.snappy.parquet
✅ Uploaded: silver/orders/part-00001-4b851e0b-6082-4265-ac6a-0ed4972b8351-c000.snappy.parquet

Total files uploaded: 2

Writing Gold tables locally...

Uploading Gold tables to S3...
✅ Uploaded: silver/gold/revenue_by_category/part-00000-66184170-b980-4f70-877c-72321e2202a6-c000.snappy.parquet

Total files uploaded: 1
✅ Uploaded: gold/orders_by_status/part-00000-4f93b180-7f6b-46f8-9efc-37b2120a0c37-c000.snappy.parquet

Total files uploaded: 1
✅ Uploaded: gold/daily_revenue/part-00000-28923b29-d114-453c-b638-8382fb95421e-c000.snappy.parquet

Total files uploaded: 1

🎉 Silver and Gold layers written to S3


In [ ]:
# Download clickstream data from S3
os.makedirs("/content/bronze/clickstream", exist_ok=True)

response = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix="bronze/clickstream/"
)

downloaded = 0
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith("/"):
        continue
    local_path = f"/content/{key}"
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(S3_BUCKET, key, local_path)
    downloaded += 1
    print(f"✅ {key}")

print(f"\nTotal clickstream files: {downloaded}")

✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142645.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142646.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142647.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142648.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142649.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142650.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142651.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142652.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142653.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142654.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142655.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142656.jsonl
✅ bronze/clickstream/date=2026-05-10/hour=14/events_20260510_142657.jsonl
✅ bronze/clickstream/date=2026-05-10/h

In [ ]:
# Read clickstream JSONL files
# JSONL = one JSON event per line
# Spark reads all files in the folder automatically
df_clickstream = spark.read \
    .option("multiline", "false") \
    .json("/content/bronze/clickstream/")

print(f"Total clickstream events: {df_clickstream.count()}")
df_clickstream.show(5)
df_clickstream.printSchema()

Total clickstream events: 486
+-------+-----------+------------+-----------+--------------------+--------------------+------------+--------------+----------+--------------------+------------+--------------------+---------+----------+----+
|browser|   category|country_code|device_type|            event_id|     event_timestamp|  event_type|    ip_address|product_id|        product_name|    referrer|          session_id|  user_id|      date|hour|
+-------+-----------+------------+-----------+--------------------+--------------------+------------+--------------+----------+--------------------+------------+--------------------+---------+----------+----+
| Safari|   Clothing|          MU|     mobile|9fede00b-b182-49a...|2026-05-10T14:27:...|product_view| 216.8.208.170| PROD-0033|Upgradable hybrid...|       email|7f247814-9fc3-4a2...|USR-00429|2026-05-10|  14|
|Firefox|Electronics|          SO|    desktop|6ce9e6ff-7379-482...|2026-05-10T14:27:...| add_to_cart| 203.76.24.164| PROD-0100|Virtual

In [ ]:
# Bronze → Silver for clickstream
# Same pattern as orders — fix types, validate, enrich

df_click_silver = df_clickstream \
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    ) \
    .withColumn("event_date", F.to_date("event_timestamp")) \
    .withColumn("event_hour", F.hour("event_timestamp"))

# Data quality — remove rows with no event_id or user_id
total_before = df_click_silver.count()

df_click_silver = df_click_silver \
    .filter(F.col("event_id").isNotNull()) \
    .filter(F.col("user_id").isNotNull()) \
    .filter(F.col("event_type").isNotNull())

total_after = df_click_silver.count()

print(f"Rows before quality check: {total_before}")
print(f"Rows after quality check:  {total_after}")
print(f"Rows dropped:              {total_before - total_after}")

# Show the conversion funnel — how many of each event type?
print("\n📊 Clickstream Event Funnel:")
df_click_silver \
    .groupBy("event_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("✅ Clickstream Bronze → Silver complete")

Rows before quality check: 486
Rows after quality check:  486
Rows dropped:              0

📊 Clickstream Event Funnel:
+----------------+-----+
|      event_type|count|
+----------------+-----+
|    product_view|  139|
|       page_view|  138|
|     add_to_cart|   83|
|  checkout_start|   54|
|        purchase|   49|
|remove_from_cart|   23|
+----------------+-----+

✅ Clickstream Bronze → Silver complete


In [ ]:
# ── Join Orders and Clickstream ───────────────────────────────────────────────
# We join on user_id — connecting what users clicked to what they bought
# This is a LEFT JOIN — keep all clickstream events
# and attach order info where it exists

df_joined = df_click_silver.join(
    df_silver.select(
        "user_id",
        "order_id",
        "total_price",
        F.col("category").alias("order_category"),  # renamed
        "status",
        "order_date"
    ),
    on="user_id",
    how="left"
)

print(f"Total joined rows: {df_joined.count()}")
df_joined.show(5)

Total joined rows: 1617
+---------+-------+-----------+------------+-----------+--------------------+--------------------+------------+-------------+----------+--------------------+--------+--------------------+----------+----+----------+----------+----------------+-----------+--------------+--------+----------+
|  user_id|browser|   category|country_code|device_type|            event_id|     event_timestamp|  event_type|   ip_address|product_id|        product_name|referrer|          session_id|      date|hour|event_date|event_hour|        order_id|total_price|order_category|  status|order_date|
+---------+-------+-----------+------------+-----------+--------------------+--------------------+------------+-------------+----------+--------------------+--------+--------------------+----------+----+----------+----------+----------------+-----------+--------------+--------+----------+
|USR-00429| Safari|   Clothing|          MU|     mobile|9fede00b-b182-49a...|2026-05-10 14:27:...|product_

In [ ]:
# ── Gold: User Journey Analysis ───────────────────────────────────────────────
# For each user — how many events did they generate and how much did they spend?
# This tells you: do highly engaged users spend more?

gold_user_journey = df_joined \
    .groupBy("user_id") \
    .agg(
        F.count("event_id").alias("total_events"),
        F.countDistinct("session_id").alias("total_sessions"),
        F.round(F.sum("total_price"), 2).alias("total_spent"),
        F.countDistinct("order_id").alias("total_orders")
    ) \
    .orderBy(F.desc("total_spent"))

print("👤 Top 10 Users by Spend:")
gold_user_journey.show(10)

# ── Gold: Device Conversion Analysis ─────────────────────────────────────────
# Which device type leads to most purchases?
# Business use: should we invest more in mobile experience?

gold_device = df_click_silver \
    .groupBy("device_type") \
    .agg(
        F.count("event_id").alias("total_events"),
        F.sum(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("total_purchases")
    ) \
    .withColumn(
        "conversion_rate",
        F.round(F.col("total_purchases") / F.col("total_events") * 100, 2)
    ) \
    .orderBy(F.desc("conversion_rate"))

print("📱 Conversion Rate by Device:")
gold_device.show()

# ── Gold: Category Interest vs Purchase ───────────────────────────────────────
# Which categories do users browse most vs actually buy?

gold_category_funnel = df_joined \
    .groupBy("order_category") \
    .agg(
        F.count("event_id").alias("total_interactions"),
        F.sum(
            F.when(F.col("status") == "completed", 1).otherwise(0)
        ).alias("completed_orders")
    ) \
    .filter(F.col("order_category").isNotNull()) \
    .orderBy(F.desc("total_interactions"))

print("🛍️ Category Interest vs Purchase:")
gold_category_funnel.show()

print("\n✅ All Gold tables complete")

👤 Top 10 Users by Spend:
+---------+------------+--------------+-----------+------------+
|  user_id|total_events|total_sessions|total_spent|total_orders|
+---------+------------+--------------+-----------+------------+
|USR-00215|          91|             1|   84415.76|           7|
|USR-00173|          60|             1|   61116.48|           5|
|USR-00047|          70|             1|    54131.0|           5|
|USR-00154|         100|             1|    53857.4|          10|
|USR-00207|          28|             1|   52884.72|           2|
|USR-00472|          88|             1|   52187.52|           4|
|USR-00357|          45|             1|    46077.0|           3|
|USR-00416|          45|             1|    44531.1|           3|
|USR-00065|          66|             1|    42631.6|           6|
|USR-00238|          32|             1|   42086.88|           4|
+---------+------------+--------------+-----------+------------+
only showing top 10 rows

📱 Conversion Rate by Device:
+---------

In [ ]:
# Write clickstream silver
df_click_silver.write \
    .mode("overwrite") \
    .parquet("/content/silver/clickstream/")

# Write gold tables
gold_user_journey.write \
    .mode("overwrite") \
    .parquet("/content/gold/user_journey/")

gold_device.write \
    .mode("overwrite") \
    .parquet("/content/gold/device_conversion/")

gold_category_funnel.write \
    .mode("overwrite") \
    .parquet("/content/gold/category_funnel/")

# Upload all silver and gold to S3
print("Uploading Silver clickstream...")
upload_folder_to_s3("/content/silver/clickstream/", "silver/clickstream/")

print("Uploading Gold tables...")
upload_folder_to_s3("/content/gold/user_journey/", "gold/user_journey/")
upload_folder_to_s3("/content/gold/device_conversion/", "gold/device_conversion/")
upload_folder_to_s3("/content/gold/category_funnel/", "gold/category_funnel/")

print("\n🎉 Phase 3 Complete!")
print("\nYour S3 data lake now has:")
print("  bronze/ → raw data exactly as ingested")
print("  silver/ → cleaned, typed, validated data")
print("  gold/   → business-ready aggregations")

Uploading Silver clickstream...
✅ Uploaded: silver/clickstream/part-00002-4128857d-af73-48de-9a3d-56db1abdad03-c000.snappy.parquet
✅ Uploaded: silver/clickstream/part-00001-4128857d-af73-48de-9a3d-56db1abdad03-c000.snappy.parquet
✅ Uploaded: silver/clickstream/part-00003-4128857d-af73-48de-9a3d-56db1abdad03-c000.snappy.parquet
✅ Uploaded: silver/clickstream/part-00000-4128857d-af73-48de-9a3d-56db1abdad03-c000.snappy.parquet

Total files uploaded: 4
Uploading Gold tables...
✅ Uploaded: gold/user_journey/part-00000-6dad84b4-a59a-4351-860b-d6e89b16b769-c000.snappy.parquet

Total files uploaded: 1
✅ Uploaded: gold/device_conversion/part-00000-28b2a54c-b60b-40ad-a04a-310339ced5e0-c000.snappy.parquet

Total files uploaded: 1
✅ Uploaded: gold/category_funnel/part-00000-4196012d-f5f0-44ce-a95c-09f049964d16-c000.snappy.parquet

Total files uploaded: 1

🎉 Phase 3 Complete!

Your S3 data lake now has:
  bronze/ → raw data exactly as ingested
  silver/ → cleaned, typed, validated data
  gold/   → 